# Training Phase 2 - Additional 90M token training data

In [ ]:
import torch
from torch import nn
import tiktoken
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
import requests

response = requests.get("https://raw.githubusercontent.com/Shanmukh-dev/Custom-GPT/refs/heads/main/model.py")
# print(response.text)
with open("model.py", "w") as f:

    f.write(response.text)
    

In [ ]:
from model import GPTConfig, CustomGPT

## Functionizing dataloader creation, Train step and Test step

### Functionize train and test dataloader creation

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SynthDataset(Dataset):
  def __init__(self, data, block_size):
    self.data = data
    self.block_size = block_size

  def __len__(self):
    return len(self.data) - self.block_size

  def __getitem__(self, idx):
    x = self.data[idx:idx+self.block_size]
    y = self.data[idx+1:idx+self.block_size+1]
    return x, y

def create_dataloaders(tokens:list, train_split:float, device:str, block_size:int, batch_size:int):
    data = torch.tensor(tokens, dtype=torch.long, device=device)
    print("Data length:", len(data))
    
    
    train_split = int(train_split*len(data))
    train_data = data[:train_split]
    test_data = data[train_split:]
    print("Train data length:", len(train_data))
    print("Test data length:", len(test_data))

    train_dataset = SynthDataset(train_data, block_size)
    test_dataset = SynthDataset(test_data, block_size)

    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)
    print("Train batches:", len(train_data)/len(train_dataloader))
    print("Test batches:", len(test_data)/len(test_dataloader))
    return train_dataloader, test_dataloader
    

### Fucntionize Train and Test step

In [ ]:
def train_step(model, train_iter, loss_fn, optimizer, scaler, device):
    try:
        X_train, y_train = next(train_iter)
    except StopIteration:
        train_iter = iter(train_dataloader)
        X_train, y_train = next(train_iter)

    X_train, y_train = X_train.to(device), y_train.to(device)
    
    model.train()
    optimizer.zero_grad()
    with autocast(device_type="cuda" if "cuda" in str(device) else "cpu"):
    
        logits = model(X_train)
        
        train_loss = loss_fn(logits.view(-1, logits.size(-1)), y_train.view(-1))
    
    scaler.scale(train_loss).backward()
    scaler.step(optimizer)
    scaler.update()
    return train_loss

def test_step(model, test_dl, n_steps):
    from tqdm.auto import tqdm
    model.eval()
    test_loss = 0
    with torch.inference_mode():

        for idx, (X_test, y_test) in tqdm(enumerate(test_dl)):
            if idx == n_steps-1:
              break
            X_test, y_test = X_test.to(device), y_test.to(device)
            test_logits = model(X_test)
            
            curr_loss = loss_fn(test_logits.view(-1, test_logits.size(-1)), y_test.view(-1))
            
            test_loss += curr_loss.item()


    test_loss = test_loss / n_steps

    return test_loss

In [ ]:
def calc_params(model):

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")

In [ ]:
config = GPTConfig(
    vocab_size=tokenizer.n_vocab, 
    block_size=256,
    d_model=256, 
    hidden_layers=1024, 
    n_heads=4, 
    n_layers=6
)

## Dataset prep

In [ ]:
from datasets import load_dataset

ds = load_dataset("PleIAs/SYNTH", split="train", streaming = True)

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
tokens = []

skip_tokens = 10_000_000
target_tokens = 90_000_000
seen_tokens = 0

for sample in ds:
    text = (
        sample["query"]
        + sample["query_seed_text"]
        + sample["synthetic_reasoning"]
        + sample["synthetic_answer"]
    )

    ids = tokenizer.encode(text)

    if seen_tokens + len(ids) <= skip_tokens:
        seen_tokens += len(ids)
        continue

    tokens.extend(ids)

    if len(tokens) >= target_tokens:
        break

print(len(tokens))
    

In [ ]:
train_dataloader, test_dataloader = create_dataloaders(tokens, 0.8, device, config.block_size, 32)

# print("Train batches:", len/len(train_dataloader))
# print(len(test_dataloader))

## Training loop (Phase 2)

In [ ]:
response = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/custom_gpt_v1.pth")

with open("custom_gpt_v1.pth", "wb") as f:
    f.write(response.content)

print("weights downloaded")

In [ ]:
from torch.amp import GradScaler, autocast

modelV2 = CustomGPT(config)
state_dict = torch.load("custom_gpt_v1.pth", map_location=device)

state_dict = {k.removeprefix("module."): v for k, v in state_dict.items()}
# print(state_dict.keys())

modelV2.load_state_dict(state_dict)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    modelV2 = nn.DataParallel(modelV2)

modelV2.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(modelV2.parameters(), lr=3e-4)
scaler = GradScaler()

In [ ]:
from tqdm.auto import tqdm
steps = 50000

torch.manual_seed(42)
train_iter = iter(train_dataloader)

for step in tqdm(range(1, steps+1)):
    modelV2.train()

    train_loss = train_step(modelV2, train_iter, loss_fn, optimizer, scaler, device)
    if step%10000 == 0:
        test_loss = test_step(modelV2, test_dataloader, 10)

        print(f"Step: {step} | Training loss: {train_loss} | Testing loss: {test_loss}")


In [ ]:
torch.save(modelV2.state_dict(), "custom_gpt_v2.pth")

# Testing the modelV2

In [4]:
import requests
import torch
from torch import nn
import requests

response = requests.get("https://raw.githubusercontent.com/Shanmukh-dev/Custom-GPT/refs/heads/main/model.py")
# print(response.text)
with open("model.py", "w") as f:

    f.write(response.text)

from model import CustomGPT, GPTConfig

config = GPTConfig(
    vocab_size=tokenizer.n_vocab, 
    block_size=256,
    d_model=256, 
    hidden_layers=1024, 
    n_heads=4, 
    n_layers=6
)

In [5]:
response = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/custom_gpt_v2.pth")

with open("custom_gpt_v2.pth", "wb") as f:
    f.write(response.content)

In [6]:
state_dict = torch.load("custom_gpt_v2.pth", map_location=device)

state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}

test_model = CustomGPT(config)

test_model.load_state_dict(state_dict)

<All keys matched successfully>

In [14]:
import tiktoken
test_model.to(device)
test_model.eval()

tokenizer = tiktoken.get_encoding("gpt2")
context = torch.tensor(tokenizer.encode("""Once upon a time, """), dtype = torch.long, device=device).unsqueeze(0)

with torch.inference_mode():
    generated = test_model.generate(context, 256)

print(tokenizer.decode(generated[0].tolist()))

Once upon a time, ichordas coloras or a infine gloshew.

Seth's interest in studio came to be driven back to a period and but is hardly surprising of symphony by his three Japanese and Pakistani songs, who published their works of his seven records. Another theory supports that many later works of the Metropolis at the Native American Museum. Al. He sought a massive and reserve of the design of schools that he could best honour the role of contradictory works. According to the ancient history, an man who existed as a major European designer in cellulis's works on many of his Jungleorate in 1956, since he won a higher Crematorium for his World Junior Padijnse botanist. By the late 1940s, the years Keynes was to expand power junctions on only 2200 men and Laoza. The Wienerżthon, Koltzany a Phécyne's own Church, would have left his private home on Bishop Tomi, in der Crassa estate ya and the military unculturatum in Kőwisto, lubhtolem Pieroogo Sozaffetan by orders of members from the upri